In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nusratjahannawal/banglapsg-v2-deduplicated/BanglaPSG_V2_Deduplicated.xlsx


In [2]:
import glob
import re, unicodedata, hashlib
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from scipy.sparse import hstack
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [3]:
hits = glob.glob("/kaggle/input/datasets/nusratjahannawal/banglapsg-v2-deduplicated/BanglaPSG_V2_Deduplicated.xlsx", recursive=True)
assert len(hits) == 1, f"Expected exactly one BanglaPSG_V2_Deduplicated.xlsx, found: {hits}"
path = hits[0]
print("Using:", path)

df = pd.read_excel(path)
assert df.columns.tolist() == ["No.", "Grievance Text", "Category", "Severity"], df.columns.tolist()
assert len(df) == 5970, len(df)
assert df["Grievance Text"].duplicated().sum() == 0, "Exact duplicate texts found: wrong file?"
df.head()

Using: /kaggle/input/datasets/nusratjahannawal/banglapsg-v2-deduplicated/BanglaPSG_V2_Deduplicated.xlsx


,No.,Grievance Text,Category,Severity
0,1,সরকারি হাসপাতাল গুলো এত অপরিষ্কার আর নোংরা।,Healthcare,High
1,2,মেডিসিন বিশেষজ্ঞ ডাঃ বসে না এসিস্ট্যান্ট দিয়ে...,Healthcare,High
2,3,রূপগঞ্জে মানুষ মারার কারখানা ইউ এস-বাংলা হাসপাতাল,Healthcare,Low
3,4,সরকারি হাসপাতালগুলোর সমস্ত টেস্টের মেশিন গুলো ...,Healthcare,Medium
4,5,সামান্য জ্বর আসলে এক হাজার নয়শত টাকার টেস্ট দয়,Healthcare,Low


In [4]:
TARGET = "Category"  

data = df[["Grievance Text", TARGET]].copy()

data.columns = ["text", "label"]

label_encoder = LabelEncoder()

data["label"] = label_encoder.fit_transform(data["label"])

print(label_encoder.classes_)

num_labels = len(label_encoder.classes_)

print("Target:", TARGET, "| Number of classes:", num_labels)

['Electricity' 'Food' 'Healthcare' 'Road & Transport' 'Waste' 'Water']
Target: Category | Number of classes: 6


In [5]:
#Leakage-safe split built from BanglaPSG_V2_Deduplicated.xlsx
import re, unicodedata, hashlib
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

_SEED = 42
_CHAR_COS, _WORD_COS, _CONTAIN, _MIN_WORDS = 0.65, 0.60, 0.90, 4
_RATIO = np.array([0.70, 0.15, 0.15])         # train / val / test

def _norm_key(t):
    t = unicodedata.normalize("NFC", str(t))
    t = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", t)
    t = t.replace("\\n", " ")
    t = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", t)
    return re.sub(r"\s+", " ", t).strip().lower()

_text = df["Grievance Text"].astype(str).map(lambda t: unicodedata.normalize("NFC", t))
_ks = _text.map(_norm_key)
_key = _ks.str.replace(" ", "", regex=False)
_strata = (df["Category"].astype(str) + " | " + df["Severity"].astype(str)).values
_n = len(df)
assert _text.duplicated().sum() == 0 and _key.duplicated().sum() == 0, "Input still has duplicates"

# 1)link near-duplicates
def _pairs(X, thr):
    out = []
    for s in range(0, _n, 500):
        sim = (X[s:s + 500] @ X.T).toarray()
        r, c = np.where(sim >= thr)
        out += [(s + a, b) for a, b in zip(r, c) if b > s + a]
    return out

_Xc = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True).fit_transform(_ks)
_Xw = TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), sublinear_tf=True).fit_transform(_ks)
_links = set(_pairs(_Xc, _CHAR_COS)) | set(_pairs(_Xw, _WORD_COS))

_B = CountVectorizer(token_pattern=r"\S+", binary=True).fit_transform(_ks).tocsr().astype(np.float32)
_nw = np.asarray(_B.sum(axis=1)).ravel()
_inter = (_B @ _B.T).tocoo()
_m = _inter.row < _inter.col
_I, _J, _V = _inter.row[_m], _inter.col[_m], _inter.data[_m]
_mn = np.minimum(_nw[_I], _nw[_J])
_sel = (_mn >= _MIN_WORDS) & (_V / _mn >= _CONTAIN - 1e-9)
_links |= set(zip(_I[_sel].tolist(), _J[_sel].tolist()))

_L = np.array(sorted(_links))
_adj = coo_matrix((np.ones(len(_L)), (_L[:, 0], _L[:, 1])), shape=(_n, _n))
_, _grp = connected_components(_adj, directed=False)
assert np.bincount(_grp).max() <= 10, "A giant linked group formed"

# 2)stratified group assignment: pure numpy + hashing
_sid = np.unique(_strata, return_inverse=True)[1]
_S, _G = _sid.max() + 1, _grp.max() + 1
_comp = np.zeros((_G, _S))
np.add.at(_comp, (_grp, _sid), 1)                       
_first = {}
for _i, _g in enumerate(_grp):
    _first.setdefault(_g, _key.iat[_i])
_tie = np.array([int(hashlib.md5(f"{_SEED}|{_first[_g]}".encode()).hexdigest()[:12], 16) for _g in range(_G)])
_order = np.lexsort((_tie, -_comp.sum(1)))              
_tgt = _RATIO[:, None] * _comp.sum(0)[None, :]          
_cur = np.zeros((3, _S))
_assign = np.zeros(_G, dtype=int)
for _g in _order:
    _new = _cur + _comp[_g]
    _cost = (((_new - _tgt) ** 2 - (_cur - _tgt) ** 2) / np.maximum(_tgt, 1)).sum(axis=1)
    _s = int(np.argmin(_cost))
    _assign[_g] = _s
    _cur[_s] = _new[_s]
_split = np.array(["train", "val", "test"])[_assign[_grp]]
df["Group"], df["Split"] = _grp, _split
_dev = np.abs(_cur - _tgt).max()
print("Worst per-stratum deviation from the 70/15/15 target (rows):", round(_dev, 1))
assert _dev <= 4, "Split is not balanced across Category x Severity"

# 3)leakage checks
for _a, _b in [("train", "val"), ("train", "test"), ("val", "test")]:
    _ma, _mb = _split == _a, _split == _b
    assert not set(_text[_ma]) & set(_text[_mb]), f"Exact text overlap {_a}/{_b}"
    assert not set(_key[_ma]) & set(_key[_mb]), f"Normalized text overlap {_a}/{_b}"
    assert not set(_grp[_ma]) & set(_grp[_mb]), f"Near-duplicate group shared by {_a}/{_b}"
assert all(_split[i] == _split[j] for i, j in _links), "A linked pair crosses splits"
assert len(set(_strata)) == len(set(_strata[_split == "train"])) == len(set(_strata[_split == "val"])) == len(set(_strata[_split == "test"]))

# 4)task-specific frames
train_df = data[df["Split"] == "train"].reset_index(drop=True)
valid_df = data[df["Split"] == "val"].reset_index(drop=True)
test_df  = data[df["Split"] == "test"].reset_index(drop=True)
assert len(train_df) + len(valid_df) + len(test_df) == len(data)
assert set(train_df["label"]) == set(valid_df["label"]) == set(test_df["label"])

print("Linked pairs:", len(_links), "| groups with >1 row:", int((np.bincount(_grp) > 1).sum()))
print("Train:", train_df.shape, "| Validation:", valid_df.shape, "| Test:", test_df.shape)
print("Split fingerprint:", hashlib.sha1(",".join(map(str, sorted(df.loc[df["Split"] == "test", "No."]))).encode()).hexdigest()[:12],
      "(must be identical in all 6 notebooks)")
print("NO LEAKAGE: exact text, normalized text and near-duplicate groups are disjoint across splits")

Worst per-stratum deviation from the 70/15/15 target (rows): 0.8
Linked pairs: 94 | groups with >1 row: 68
Train: (4176, 2) | Validation: (899, 2) | Test: (895, 2)
Split fingerprint: 9dc9e17dd435 (must be identical in all 6 notebooks)
NO LEAKAGE: exact text, normalized text and near-duplicate groups are disjoint across splits


In [6]:
EXPECTED_FINGERPRINT = "9dc9e17dd435"
fingerprint = hashlib.sha1(",".join(map(str, sorted(df.loc[df["Split"] == "test", "No."]))).encode()).hexdigest()[:12]
assert fingerprint == EXPECTED_FINGERPRINT, f"Split differs from the transformer notebooks: {fingerprint}"
assert (len(train_df), len(valid_df), len(test_df)) == (4176, 899, 895)
print("Split matches the transformer notebooks:", fingerprint, "| train/val/test =", len(train_df), len(valid_df), len(test_df))

Split matches the transformer notebooks: 9dc9e17dd435 | train/val/test = 4176 899 895


In [7]:
# Same input for every SVM run:punctuation-stripped normalized text
train_txt = train_df["text"].map(_norm_key).values
valid_txt = valid_df["text"].map(_norm_key).values
test_txt  = test_df["text"].map(_norm_key).values

y_train = train_df["label"].values
y_valid = valid_df["label"].values
y_test  = test_df["label"].values

#Vectorizers are fitted on the TRAIN split only,then applied to validation andtest
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2)
word_vec = TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), sublinear_tf=True, min_df=2)

X_train = hstack([char_vec.fit_transform(train_txt), word_vec.fit_transform(train_txt)]).tocsr()
X_valid = hstack([char_vec.transform(valid_txt), word_vec.transform(valid_txt)]).tocsr()
X_test  = hstack([char_vec.transform(test_txt),  word_vec.transform(test_txt)]).tocsr()

print("Feature matrix (train):", X_train.shape)
print("Feature matrix (valid):", X_valid.shape)
print("Feature matrix (test): ", X_test.shape)

Feature matrix (train): (4176, 51071)
Feature matrix (valid): (899, 51071)
Feature matrix (test):  (895, 51071)


In [8]:
C_GRID = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]

rows = []
for C in C_GRID:
    clf = LinearSVC(C=C, max_iter=20000, random_state=42)
    clf.fit(X_train, y_train)
    p = clf.predict(X_valid)
    rows.append({"C": C,
                 "val_accuracy": accuracy_score(y_valid, p),
                 "val_macro_f1": f1_score(y_valid, p, average="macro")})

tuning = pd.DataFrame(rows)
print(tuning.to_string(index=False))

best_C = float(tuning.sort_values(["val_macro_f1", "C"], ascending=[False, True]).iloc[0]["C"])
print("\nBest C (by validation macro-F1):", best_C)

    C  val_accuracy  val_macro_f1
 0.01      0.951057      0.951021
 0.03      0.954394      0.954386
 0.10      0.952169      0.952147
 0.30      0.949944      0.950013
 1.00      0.953281      0.953416
 3.00      0.952169      0.952272
10.00      0.948832      0.948950

Best C (by validation macro-F1): 0.03


In [9]:
svm = LinearSVC(C=best_C, max_iter=20000, random_state=42)
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)
y_true = y_test

result = {
    "accuracy":  accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, average="macro"),
    "recall":    recall_score(y_true, y_pred, average="macro"),
    "f1":        f1_score(y_true, y_pred, average="macro"),
}
print("TEST:", {k: round(v, 4) for k, v in result.items()})
print()
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

TEST: {'accuracy': 0.9385, 'precision': 0.9393, 'recall': 0.9385, 'f1': 0.9387}

                  precision    recall  f1-score   support

     Electricity       0.97      0.99      0.98       149
            Food       0.91      0.95      0.93       155
      Healthcare       0.97      0.93      0.95       146
Road & Transport       0.92      0.92      0.92       154
           Waste       0.93      0.92      0.92       149
           Water       0.94      0.93      0.94       142

        accuracy                           0.94       895
       macro avg       0.94      0.94      0.94       895
    weighted avg       0.94      0.94      0.94       895



In [10]:
cm = confusion_matrix(y_true, y_pred, labels=range(num_labels))
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{c}" for c in label_encoder.classes_],
    columns=[f"pred_{c}" for c in label_encoder.classes_]
)
print("Confusion Matrix:")
print(cm_df.to_string())

test_ids = df.loc[df["Split"] == "test", "No."].values
assert len(test_ids) == len(y_pred)

predictions = pd.DataFrame({
    "No.": test_ids,
    "y_true": label_encoder.classes_[y_true],
    "y_pred": label_encoder.classes_[y_pred],
})
predictions.to_excel(f"./SVM_{TARGET}_test_predictions.xlsx", index=False)

summary = pd.DataFrame([{"Model": "LinearSVC (TF-IDF char+word)", "Task": TARGET, "Best C": best_C,
                         "Accuracy": result["accuracy"], "Macro Precision": result["precision"],
                         "Macro Recall": result["recall"], "Macro F1": result["f1"]}])
with pd.ExcelWriter(f"./SVM_{TARGET}_results.xlsx") as w:
    summary.to_excel(w, sheet_name="summary", index=False)
    tuning.to_excel(w, sheet_name="validation_tuning", index=False)
    cm_df.to_excel(w, sheet_name="confusion_matrix")

print("\nSaved:", f"SVM_{TARGET}_test_predictions.xlsx", "and", f"SVM_{TARGET}_results.xlsx")

Confusion Matrix:
                       pred_Electricity  pred_Food  pred_Healthcare  pred_Road & Transport  pred_Waste  pred_Water
true_Electricity                    147          0                0                      1           1           0
true_Food                             0        147                1                      1           4           2
true_Healthcare                       1          6              136                      3           0           0
true_Road & Transport                 2          5                2                    141           3           1
true_Waste                            0          0                1                      6         137           5
true_Water                            2          3                0                      2           3         132

Saved: SVM_Category_test_predictions.xlsx and SVM_Category_results.xlsx
